### 基于K-Means聚类与Prophet时序预测的重庆上市车企财务智能预警研究

#### ——赛力斯外部协同经营模式对标长安自主研发模式

一、数据获取
1. 获取财务报表数据：2016年Q1-2025年Q4
* `data/collect_financial_data.py` # 获取财务数据
* 获取8家对标车企（赛力斯★/长安汽车★/比亚迪/上汽集团/长城汽车/广汽集团/江淮汽车/北汽蓝谷）的资产负债表、利润表和现金流量表的年度数据，并保存为excel文件。
2. 获取车企销售数据：2023年Q1-2026年Q4
* `data/add_company_sales.py` # 添加公司销售数据
* `data/extract_sales_structure.py` # 获取销售结构数据

二、数据探索

1. 财务指标

（1）数据读取   

In [4]:
import pandas as pd
from pathlib import Path

data_dir = Path('data')
all_data = {
    file.stem.replace('_财务数据', ''): pd.read_excel(file)
    for file in data_dir.glob('*_财务数据.xlsx')
}
# 加载
for company, df in all_data.items():
    print(f"{company}: {len(df)} 条记录")
list(all_data.values())[0].head() # 预览

000625_长安汽车: 69 条记录
002594_比亚迪: 64 条记录
600104_上汽集团: 69 条记录
600418_江淮汽车: 61 条记录
600733_北汽蓝谷: 61 条记录
601127_赛力斯: 63 条记录
601238_广汽集团: 69 条记录
601633_长城汽车: 65 条记录


,Unnamed: 0,2016年报,2016Q1,2016Q2,2016Q3,2017年报,2017Q1,2017Q2,2017Q3,2018年报,...,2023Q2,2023Q3,2024年报,2024Q1,2024Q2,2024Q3,2025年报,2025Q1,2025Q2,2025Q3
0,货币资金,2.478300e+10,1.871700e+10,2.135800e+10,2.510300e+10,2.263200e+10,2.832800e+10,2.788200e+10,2.399200e+10,9.981000e+09,...,6.598600e+10,7.219200e+10,6.418200e+10,7.006800e+10,7.093900e+10,7.000500e+10,5.402200e+10,6.012900e+10,5.283000e+10,5.524100e+10
1,交易性金融资产,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.620000e+08,1.570000e+08,1.550000e+08,1.440000e+08,1.530000e+08,1.670000e+08,NaN,1.500000e+08,1.620000e+08,1.024000e+05
2,应收账款,1.499000e+09,1.270000e+09,1.480000e+09,1.518000e+09,1.807000e+09,2.427000e+09,1.900000e+09,1.996000e+09,1.409000e+09,...,2.477000e+09,2.669000e+09,3.398000e+09,3.277000e+09,2.989000e+09,4.226000e+09,4.197000e+09,5.021000e+09,6.628000e+09,7.216000e+09
3,应收票据及应收账款,3.050100e+10,2.414100e+10,2.131300e+10,2.162200e+10,3.096300e+10,2.588000e+10,2.363500e+10,2.336100e+10,2.197100e+10,...,3.644100e+10,3.804700e+10,4.897900e+10,3.698200e+10,3.698500e+10,3.344300e+10,3.719500e+10,4.285000e+10,3.218500e+10,2.773400e+10
4,预付款项,1.061000e+09,6.700000e+08,9.910000e+08,8.800000e+08,1.102000e+09,1.323000e+09,1.274000e+09,1.832000e+09,8.660000e+08,...,6.620000e+08,9.880000e+08,5.070000e+08,4.820000e+08,5.830000e+08,4.560000e+08,4.920000e+08,4.360000e+08,4.000000e+08,7.940000e+08


(2) 报表合并

In [9]:
def integrate_financial_data(company_code):          # 参数是 company_code，不是 name
    file_path = f'data/{company_code}_财务数据.xlsx'
    
    df_balance = pd.read_excel(file_path, sheet_name='资产负债表', index_col=0)
    df_income  = pd.read_excel(file_path, sheet_name='利润表', index_col=0)
    df_cash    = pd.read_excel(file_path, sheet_name='现金流量表', index_col=0)
    
    df_balance = df_balance.T.add_suffix('_资产')
    df_income  = df_income.T.add_suffix('_利润')
    df_cash    = df_cash.T.add_suffix('_现金流')
    
    data = pd.concat([df_income, df_balance, df_cash], axis=1).fillna(0)
    print(f'完成: {len(data)} 期 × {len(data.columns)} 个指标')
    return data  

# 查看
companies = [f.stem.replace('_财务数据','') for f in Path('data').glob('*_财务数据.xlsx')]
print('可用公司:', companies)

data = integrate_financial_data('601127_赛力斯')
data.info()
data.head()
#data.to_csv('data/601127_赛力斯_财务数据_merged.csv', index=True, encoding='utf-8-sig')  # 保存为 CSV 文件

可用公司: ['000625_长安汽车', '002594_比亚迪', '600104_上汽集团', '600418_江淮汽车', '600733_北汽蓝谷', '601127_赛力斯', '601238_广汽集团', '601633_长城汽车']
完成: 40 期 × 154 个指标
<class 'pandas.DataFrame'>
Index: 40 entries, 2016年报 to 2025Q3
Columns: 154 entries, 一、营业总收入_利润 to 间接法-现金及现金等价物净增加额_现金流
dtypes: float64(154)
memory usage: 48.4+ KB


,一、营业总收入_利润,营业收入_利润,税金及附加_利润,销售费用_利润,管理费用_利润,研发费用_利润,财务费用_利润,其中：利息费用_利润,其中：利息收入_利润,投资收益_利润,...,投资损失_现金流,递延所得税资产减少_现金流,递延所得税负债增加_现金流,存货的减少_现金流,经营性应收项目的减少_现金流,经营性应付项目的增加_现金流,间接法-经营活动产生的现金流量净额_现金流,现金的期末余额_现金流,减：现金的期初余额_现金流,间接法-现金及现金等价物净增加额_现金流
2016年报,1.619243e+10,1.619243e+10,5.070655e+08,1.040867e+09,8.576498e+08,0.000000e+00,31223587.42,0.000000e+00,0.000000e+00,17085648.52,...,-17085600.0,-2747300.0,-2803400.0,-398000000.0,-3.166000e+09,3.396000e+09,1.146000e+09,2.354000e+09,7.830000e+08,1.571000e+09
2016Q1,3.192494e+09,3.192494e+09,8.975824e+07,2.354520e+08,1.904555e+08,0.000000e+00,19150137.50,0.000000e+00,0.000000e+00,3113381.63,...,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2016Q2,6.335556e+09,6.335556e+09,1.679867e+08,4.499726e+08,3.629218e+08,0.000000e+00,39083090.27,0.000000e+00,0.000000e+00,4811555.19,...,-4811600.0,2414900.0,-1409500.0,-94636200.0,-9.450730e+07,2.320000e+08,6.140000e+08,1.813000e+09,7.830000e+08,1.030000e+09
2016Q3,1.020144e+10,1.020144e+10,2.830919e+08,7.015685e+08,5.480660e+08,0.000000e+00,49897945.70,0.000000e+00,0.000000e+00,13988970.02,...,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2017年报,2.193376e+10,2.193376e+10,6.996699e+08,1.282389e+09,9.138462e+08,4.678123e+08,30211619.41,1.378618e+08,1.179849e+08,14875439.06,...,-14875400.0,-7041700.0,-2344900.0,-242000000.0,-9.310000e+08,1.340000e+08,9.970000e+08,4.366000e+09,2.354000e+09,2.012000e+09


(3) 插到新table

In [10]:
from pathlib import Path
import pandas as pd

def integrate_financial_data(company_code):
    file_path = f'data/{company_code}_财务数据.xlsx'
    df_balance = pd.read_excel(file_path, sheet_name='资产负债表', index_col=0)
    df_income  = pd.read_excel(file_path, sheet_name='利润表', index_col=0)
    df_cash    = pd.read_excel(file_path, sheet_name='现金流量表', index_col=0)
    df_balance = df_balance.T.add_suffix('_资产')
    df_income  = df_income.T.add_suffix('_利润')
    df_cash    = df_cash.T.add_suffix('_现金流')
    return pd.concat([df_income, df_balance, df_cash], axis=1).fillna(0)

# 逐家公司追加合并报表 sheet
for fp in Path('data').glob('*_财务数据.xlsx'):
    code = fp.stem.replace('_财务数据', '')
    data = integrate_financial_data(code)
    with pd.ExcelWriter(fp, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        data.to_excel(writer, sheet_name='合并报表', index=True)
    print(f'✅ {code}: 已追加合并报表 ({len(data)}期×{len(data.columns)}指标)')


✅ 000625_长安汽车: 已追加合并报表 (40期×160指标)
✅ 002594_比亚迪: 已追加合并报表 (40期×159指标)
✅ 600104_上汽集团: 已追加合并报表 (40期×167指标)
✅ 600418_江淮汽车: 已追加合并报表 (40期×155指标)
✅ 600733_北汽蓝谷: 已追加合并报表 (40期×149指标)
✅ 601127_赛力斯: 已追加合并报表 (40期×154指标)
✅ 601238_广汽集团: 已追加合并报表 (40期×169指标)
✅ 601633_长城汽车: 已追加合并报表 (40期×160指标)


（4） 静态财务指标
* 盈利能力、营运能力、偿债能力、成长能力

In [ ]:
import numpy as np

def financial_ratio(name, data):
    """计算15个财务指标并保存到Excel新sheet"""
    # 列名映射（合并后带 _利润/_资产 后缀）
    col = {
        '营收': '营业收入_利润',
        '营成': '营业成本_利润',
        '营利': '营业利润_利润',
        '净利': '净利润_利润',
        '归母净利': '归属于母公司所有者的净利润_利润',
        '利息': '其中：利息费用_利润',
        '存货': '存货_资产',
        '资产': '资产合计_资产',
        '负债': '负债合计_资产',          
        '货币资金': '货币资金_资产',       
        '应收': '应收账款_资产',
        '流资': '流动资产合计_资产',
        '流负': '流动负债合计_资产',
        '预付': '预付款项_资产',
        '权益': '归属于母公司所有者权益合计_资产',
    }
    
    # 息税前利润
    data['息税前利润'] = data[col['营利']] + data[col['利息']]
    
    # 盈利能力（4）
    data['毛利率'] = 1 - data[col['营成']] / data[col['营收']]
    data['营业利润率'] = data[col['营利']] / data[col['营收']]
    data['净利润率'] = data[col['净利']] / data[col['营收']]
    data['净资产收益率'] = 2 * data[col['归母净利']] / (
        data[col['权益']] + data[col['权益']].shift(1))
    
    # 营运能力（3）
    data['存货周转率'] = 2 * data[col['营成']] / (
        data[col['存货']] + data[col['存货']].shift(1))
    data['总资产周转率'] = 2 * data[col['营收']] / (
        data[col['资产']] + data[col['资产']].shift(1))
    data['应收账款周转率'] = 2 * data[col['营收']] / (
        data[col['应收']] + data[col['应收']].shift(1))
    
    # 偿债能力（5 ← 原来3个 + 新增2个）
    data['流动比率'] = data[col['流资']] / data[col['流负']]
    data['速动比率'] = (data[col['流资']] - data[col['存货']] - data[col['预付']]) / data[col['流负']]
    data['利息保障倍数'] = data['息税前利润'] / data[col['利息']]
    data['资产负债率'] = data[col['负债']] / data[col['资产']]         
    data['货币资金占比'] = data[col['货币资金']] / data[col['资产']]   
    
    # 成长能力（3）
    data['营业收入增长率'] = data[col['营收']] / data[col['营收']].shift(1) - 1
    data['营业利润增长率'] = data[col['营利']] / data[col['营利']].shift(1) - 1
    data['净利润增长率'] = data[col['净利']] / data[col['净利']].shift(1) - 1
    
    # 提取15个指标
    cols = ['毛利率','营业利润率','净利润率','净资产收益率',
            '存货周转率','总资产周转率','应收账款周转率',
            '流动比率','速动比率','利息保障倍数','资产负债率','货币资金占比',
            '营业收入增长率','营业利润增长率','净利润增长率']
    df = data[cols].iloc[1:].copy()
    df.index = data.index[1:]
    
    # 无穷大转0
    df[np.isinf(df)] = 0
    
    # 保存
    with pd.ExcelWriter(f'data/{name}_财务数据.xlsx',
                        engine='openpyxl',
                        mode='a',
                        if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name='财务指标表', index=True)
    
    print(f'✅ {name}: 财务指标表已保存 ({len(df)}期×{len(df.columns)}指标)')
    return df


# ============ 使用 ============
data = integrate_financial_data('601127_赛力斯')
df_ratio = financial_ratio(name='601127_赛力斯', data=data)
df_ratio.head()


✅ 601127_赛力斯: 财务指标表已保存 (39期×15指标)


,毛利率,营业利润率,净利润率,净资产收益率,存货周转率,总资产周转率,应收账款周转率,流动比率,速动比率,利息保障倍数,资产负债率,货币资金占比,营业收入增长率,营业利润增长率,净利润增长率
2016Q1,0.043866,0.044842,0.042124,0.035970,5.595694,0.188598,16.246791,NaN,NaN,0.000000,0.762251,0.000000,-0.802840,-0.799713,-0.788253
2016Q2,0.035776,0.036536,0.034175,0.062562,15.028030,0.433749,30.241316,0.906216,0.810236,0.000000,0.726142,0.196196,0.984516,0.616925,0.610027
2016Q3,0.031950,0.033321,0.031242,0.081030,11.777589,0.629952,24.700825,0.902486,0.792878,0.000000,0.752065,0.191288,0.610189,0.468511,0.472005
2017年报,0.057857,0.063177,0.050213,0.173452,18.769072,1.072346,32.160944,0.996867,0.882267,11.051469,0.754407,0.247154,1.150065,3.076576,2.455630
2017Q1,0.059936,0.060224,0.048794,0.045958,3.692621,0.260812,8.833686,0.894994,0.770151,0.000000,0.772137,0.225941,-0.732175,-0.744695,-0.739740
